# Self-driving PFM: follow ferroelectric domain walls

The same machinery as `Run_SelfDriving_Scan.ipynb` (topography), with the **domain-wall
explorer** (`MODALITY = "domain_wall_explorer"`, `selfdriving_afm/pfm.py`): it explores the
area and follows every **domain wall** it crosses -- where the **PFM phase flips** between
the two domains (ideally 180 deg; any separation of at least `min_phase_separation_deg`) and
the **PFM amplitude dips** -- until the wall closes or reaches the area's border, then explores
again.

**Asylum Research software in PFM mode** (contact, the drive on while engaged). The setup
(3a) copies the PFM signals to BNC outputs, like the X/Y sensors (a PIDS loop as a straight
copy -> a free DAC -> a BNC, locked), and checks each AI reads its copy:

| signal | default route | PyScanner |
|---|---|---|
| PFM amplitude (`Amplitude`) | Cypher loop 5 -> OutFastA -> BNCOut2 | **AI0** |
| PFM phase (`Phase`) | **ARC** loop 2 -> ARC OutA -> ARC BNCOut0 | **AI3** |
| X / Y sensors | Cypher loops 3 / 4 -> OutC / OutD -> BNCOut0 / 1 | AI1 / AI2 |

Either PFM signal can be switched off (`cfg.pfm.amplitude.enabled`, `cfg.pfm.phase.enabled`):
- **both**: domains from the phase, every wall confirmed by an amplitude dip (most robust);
- **phase only**: domains from the phase;
- **amplitude only**: the wall is where the amplitude drops below `amp_dip_fraction` x the
  domains' -- followed round as a thin strip.

The live view, the saved file and the reference raster show the **phase** (or the amplitude
if the phase is off) as "Z", in degrees (V); both signals are saved (`ch_amplitude`,
`ch_phase`). Everything topography-specific is unchanged in the other notebook.

**CONFIRM ON HARDWARE** (first run): the ARC's DAC / BNC names (`OutA`, `BNCOut0`) and that its
loops accept `Phase` as an input -- the setup prints the copy check ("ARC OutA = ... (Phase ...)")
and whether AI3 reads it.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # sklearn GP kernel-bound convergence warnings are benign

import time
import pathlib
import numpy as np
import matplotlib
%matplotlib widget
import matplotlib.pyplot as plt

from selfdriving_afm.config import ExperimentConfig
from selfdriving_afm.execution import ContinuousExecutor
from selfdriving_afm.planners.edge_follow_planner import EdgeFollowPlanner
from selfdriving_afm.planners.gp_path_planner import GPPathPlanner
from selfdriving_afm.planners.edge_explorer_planner import EdgeExplorerPlanner
from selfdriving_afm.planners.grid_refine_planner import GridRefinePlanner
from selfdriving_afm.viz import (LiveEdgeFollowView, LiveGPPathView, LiveGridRefineView, build_grid,
                                  reconstruct_topography, subsample_for_reconstruction,
                                  bounds_from_history, grid_from_bounds)
from selfdriving_afm.reconstruction import reconstruct_image
from selfdriving_afm import modes  # the scan modes -- shared with Simulation.ipynb


## 1. Backend: real hardware or simulation

Set `USE_HARDWARE = True` to connect to PyScanner over the same BEPyAE/ActiveX
bridge used elsewhere on this instrument. Leave it `False` to rehearse against
`SimulateBackend` first -- same planner code either way.

The PyScanner `exe_path`/`vi_path` come from `hardware_paths.json` in this package's
root, next to this notebook -- **not** typed into the notebook, since that install
path is different on every microscope's computer. First time on a new computer:
copy `hardware_paths.example.json` to `hardware_paths.json` and edit those two paths
for this machine; it's gitignored, so it never gets overwritten by this notebook or
shared between computers.


In [ ]:
USE_HARDWARE = True  # <-- flip to True on the instrument

if USE_HARDWARE:
    import json
    import pathlib
    from selfdriving_afm.backends.pyscanner_backend import connect, PyScannerBackend

    HARDWARE_PATHS_FILE = pathlib.Path("hardware_paths.json")  # per-computer, gitignored
    if not HARDWARE_PATHS_FILE.exists():
        raise FileNotFoundError(
            f"{HARDWARE_PATHS_FILE.resolve()} not found. Copy hardware_paths.example.json "
            f"to hardware_paths.json (next to this notebook) and edit exe_path/vi_path "
            f"for THIS computer -- every microscope's PyScanner install lives somewhere different."
        )
    hw_paths = json.loads(HARDWARE_PATHS_FILE.read_text())

    vi = connect(exe_path=hw_paths["exe_path"], vi_path=hw_paths["vi_path"])
    # the Asylum Research software (Igor Pro), over COM: engage / withdraw, crosspoint,
    # X/Y loops (section 3a)
    from selfdriving_afm import asylum
    igor = asylum.connect_igor()
else:
    from selfdriving_afm.backends.simulate_backend import SimulateBackend, default_topography
    from selfdriving_afm import asylum
    igor = None   # no Igor: section 3a prints the commands without sending them


## 2. Configuration

`cfg.execution.start_xy_V` is fixed at `(0.0, 0.0)` per spec -- every run starts there.
Tune the rest from your simulation results before pointing this at an unknown sample.

The area to explore and the expected feature sizes are **not** set here: they are
the priors, asked in section 2b right after this cell.


In [ ]:
cfg = ExperimentConfig()

# --- where this run's data gets saved -- created now if it doesn't exist yet, so a
# permissions problem shows up here rather than after a long scan finishes ---
OUTPUT_DIR = pathlib.Path("scan_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODALITY = "domain_wall_explorer"  # "edge_follow", "gp_path", "edge_explorer" or "grid_refine"  <-- TOGGLE SELF-DRIVING MODE HERE

#
# --- tip velocity: the ONE knob that sets how fast the tip moves, everywhere
# (spirals, circles, GP path segments, raster/Lissajous/spiral scans incl. raster
# turnarounds, moves between scans, and the return to origin). It is the real speed
# only if sample_rate_hz = PyScanner's data_rate -- checked when the executor is built.
# Point density is derived from this + sample_rate_hz, not set separately -- and it
# scales as arc_length * sample_rate_hz / tip_velocity_V_s, so a slow tip on a fast
# DAQ over a large search area can need tens of millions of samples for one spiral
# (trajectory.py raises a clear error above 10,000,000 points/primitive rather than
# hanging -- if you hit that, raise tip_velocity_V_s, lower sample_rate_hz, or
# shrink max_search_radius_V/path_segment_length_V).
#
# The values below are chosen to run FAST IN SIMULATION against this project's small
# volt-scale synthetic sample -- they are NOT a real tip speed or DAQ rate. Before a
# real run, replace both with your actual instrument numbers (CONFIRM ON HARDWARE --
# see README "Tip velocity"); real hardware will likely want a much slower
# tip_velocity_V_s than this.
cfg.velocity.tip_velocity_V_s = 3
cfg.velocity.sample_rate_hz = 1e6 #1.0e6 or 0.1e6 or 0.01e6

# --- edge-follow knobs (only used if MODALITY == "edge_follow") ---
# search spirals: the FIRST one covers the whole area to explore, and the spiral turn
# spacing and the tracing-circle sizes come from the feature sizes -- all set by the
# priors (section 2b). Every edge a spiral crosses gets traced; then the next spiral goes
# to the nearest area not scanned yet (or grows -- see spiral_search_mode below)
cfg.edge_follow.spiral_growth_factor = 1.5        # how much bigger each retry spiral gets ("grow" mode only)
cfg.edge_follow.max_search_radius_V = 4.0         # give up growing the spiral past this ("grow" mode only)
# what a spiral that finds nothing new does next:
cfg.edge_follow.spiral_search_mode = "relocate"          # "relocate" = next spiral (same radius) goes to the
                                                          #   nearest area not scanned yet; "grow" = old behaviour
                                                          #   (bigger spiral, same center)
cfg.edge_follow.first_scan_pattern = "spiral"             # FIRST search: "spiral" (the circle inside the area), or
                                                          #   "raster" / "lissajous" over the whole square area, corners
                                                          #   included (lines 0.5 x smallest feature apart, from the
                                                          #   priors); later searches are always spirals
cfg.edge_follow.explore_min_unexplored_fraction = 0.5     # a new spiral spot must be at least this much unexplored
                                                          #   (lower = spirals overlap more but miss less)
cfg.edge_follow.spiral.n_cycles = 6             # how tightly wound the spiral is (the priors
                                                  #   replace this with turns 0.5 x smallest feature apart)
cfg.edge_follow.spiral.direction = "ccw"          # "ccw" or "cw"
# edge detection -- what counts as "found an edge" while spiraling/tracing
cfg.edge_follow.edge_detect.mode = "manual"           # "auto_mad" or "manual" + manual_threshold
cfg.edge_follow.edge_detect.manual_threshold = 100     # required if mode == "manual"
cfg.edge_follow.edge_detect.mad_k = 6.0                 # higher = less sensitive (fewer false edges)
# tracing circles, once the spiral finds an edge -- their radius comes from the priors
# (section 2b: sized from the measured edge curvature, between limits from the feature sizes)
cfg.edge_follow.circle.overlap_fraction = 0.6
# one edge = one feature: what happens when a tracing circle sees a SEPARATE edge
cfg.edge_follow.on_new_edge = "switch"          # "switch" = trace the new edge now, then come back
                                                  #   and finish this one; "queue" = finish this one first
cfg.edge_follow.trace_both_directions = True     # edge lost one way -> follow it the other way too (same feature)
cfg.edge_follow.same_feature_distance_V = None   # closer than this to a traced edge = same edge
                                                  #   (None = a quarter of the circle radius, measured from the traced edge line)
cfg.edge_follow.max_circles_per_feature = 150    # per direction along an edge
cfg.edge_follow.edge_detect.min_run_V = 0.015    # on/off stretches shorter than this are noise, not edges
#
# --- stopping criteria: several apply at once, whichever fires first wins. Toggle
# any of them off by setting it to None. See README "When does a scan stop?" for
# the full reasoning behind each one.
cfg.edge_follow.max_features = 6           # (1) stop once this many features are traced
                                             #     ("search-exhausted" -- no edge found even
                                             #     after growing to max_search_radius_V -- is
                                             #     always on too, it's how the planner gives up)
cfg.edge_follow.max_run_time_s = None      # (2) TIME: wall-clock cap in seconds, e.g. 120 = 2
                                             #     minutes. Set to None to disable and rely only
                                             #     on max_features / search exhaustion above.

# --- GP path knobs (only used if MODALITY == "gp_path") ---
# step length -- how far each committed path segment goes
cfg.gp_path.initial_path_length_V = 0.5    # length of the very FIRST segment (no GP fit
                                              # exists yet to pick a direction from, so this
                                              # one is just a fixed-length probe)
cfg.gp_path.path_segment_length_V = 0.5    # length of every segment AFTER the first --
                                              # THIS is "the GP step size" -- bigger covers
                                              # ground faster but coarser gradient estimates;
                                              # smaller is finer but needs more segments
# direction search -- how the planner picks where each segment goes
cfg.gp_path.n_candidate_directions = 8     # how many directions are scored at each step
                                              # (evenly spaced) before committing to the best
cfg.gp_path.exploration_weight = 0.5        # candidate score = gradient + this * GP uncertainty
                                              # -- higher favors exploring unknown area over
                                              # following the steepest known gradient
cfg.gp_path.gradient_eval_resolution_V = 0.005  # internal only -- how finely candidate
                                              # directions are sampled against the fitted GP.
                                              # Never touches hardware; no need to change this
# GP refit -- exact GP regression is O(n^3), so re-fitting every segment gets slow on a
# long run; see README "The GP path planner in detail" for the full reasoning
cfg.gp_path.refit_every = 5                 # re-fit the GP every N segments instead of every one
cfg.gp_path.gp_n_restarts_optimizer = 0     # 0 = fast (fixed init hyperparams); raise once
                                              # you're tuning against real data for a better fit
#
# --- stopping criteria: several apply at once, whichever fires first wins. Toggle
# any of them off by setting it to None. See README "When does a scan stop?" for
# the full reasoning behind each one.
cfg.gp_path.max_segments = 60               # (1) hard cap, always enforced regardless of the below
cfg.gp_path.max_run_time_s = 1020.0          # (2) TIME: wall-clock cap in seconds, e.g. 120 = 2
                                             #     minutes. Set to None to disable.
cfg.gp_path.target_mean_uncertainty = None  # (3) "good enough": stop once the GP's mean
                                             #     uncertainty over stop_check_extent_V drops to
                                             #     this (same units as z). None disables it --
                                             #     set a value once you know a reasonable noise
                                             #     floor from simulation, e.g. 3.0
cfg.gp_path.stop_check_extent_V = 0.3        # required if target_mean_uncertainty is set
cfg.gp_path.stop_check_grid_n = 30           # resolution of the grid that uncertainty check runs on
cfg.gp_path.min_segments_before_stop_check = 8  # don't even check (3) until this many segments
                                                  # are in, so a lucky-quiet early patch can't
                                                  # end the run prematurely

# --- edge-explorer knobs (only used if MODALITY == "edge_explorer") ---
# (area, first path, exploration spacing and zigzag step range: set by the priors, section 2b)
# a continuous path of straight segments: EXPLORES until it crosses an edge, then FOLLOWS
# that edge (a zigzag across it) until it's closed -- or lost both ways -- then explores
# again; other edges crossed meanwhile are queued and traced next. Closed edges are not
# re-scanned. Only a few ms of computing between segments.
cfg.edge_explorer.initial_path_length_V = 1.0   # the first straight path
cfg.edge_explorer.initial_direction_deg = 0.0   # its direction (0 = +X, 90 = +Y)
cfg.edge_explorer.segment_length_V = 0.5        # step size while exploring
cfg.edge_explorer.follow_recovery_tries = 3      # following: a zigzag step missed the edge (a corner) -> turn
cfg.edge_explorer.follow_recovery_turn_deg = 60.0  #   this much and retry, this many times, before it's lost
cfg.edge_explorer.max_segments_per_feature = 500  # following: safety cap per direction along an edge
# following style: "segments" = zigzag of straight segments across the edge (default);
# "weave" = a smooth sine weaving back and forth across it (no corners, easier on the piezo),
# re-centred and re-aimed on the edge after every waveform (see README "smooth edge following")
cfg.edge_explorer.follow_style = "weave"     # "segments" or "weave"
cfg.edge_explorer.follow_weave_amplitude_V = 0.065   # weave: how far either side of the edge (None = half the
                                                 #   shortest follow step, >= 0.2 V -- wider than the piezo error)
cfg.edge_explorer.follow_weave_wavelength_V = 0.06  # weave: distance along the edge per sine cycle (None = same rule)
cfg.edge_explorer.follow_weave_cycles = 3.0      # weave: sine cycles per waveform (then re-fitted to its crossings)
cfg.edge_explorer.follow_weave_corner_deg = 45.0  # weave: crossings stopped part-way (a corner) -> restart turned this much
cfg.edge_explorer.scan_interior = True          # once an edge closes: parallel lines across its inside (one
                                                 #   waveform) -- measures the top, finds holes (followed too)
cfg.edge_explorer.interior_max_lines = 25       # at most this many lines per feature (their spacing and margin
                                                 #   come from the priors)
cfg.edge_explorer.explore_scale_V = None        # exploring sweeps ~this far apart = smallest feature you
                                                 #   don't want to miss (None = segment_length_V); bigger = faster
                                                 #   (the priors set it to the smallest feature)
cfg.edge_explorer.explore_lookahead_V = 0.4      # each direction is scored this far ahead (None = 4 x
                                                 #   segment_length_V): bigger = heads for big empty regions
cfg.edge_explorer.min_score = 0.02                # when nothing nearby is more unexplored than this, jump to
                                                 #   the nearest unexplored region (0.02 = squeeze into thin gaps)
cfg.edge_explorer.straight_bonus = 0.3           # preference for going straight on: longer sweeps (0.1 = curlier)
cfg.edge_explorer.loop_closure_min_coverage_deg = 300.0  # closed = back at the start, crossings this far round
cfg.edge_explorer.edge_detect.mode = "auto_mad"  # "auto_mad" or "manual" + manual_threshold
cfg.edge_explorer.edge_detect.manual_threshold = None
cfg.edge_explorer.edge_detect.mad_k = 6.0
# stopping: whichever comes first, checked BETWEEN features -- the edge being followed is
# always finished first (it also stops once the whole region has been explored)
cfg.edge_explorer.max_segments = 1500
cfg.edge_explorer.max_features = 25             # closed edges
cfg.edge_explorer.max_run_time_s = 2000.0

# --- grid + refine knobs (only used if MODALITY == "grid_refine") ---
# (area and sparse line spacing: set by the priors, section 2b)
# 1. ONE sparse scan of the whole area -> 2. fast inpainting -> 3. find where the data shows
# structure (local height variation with the local tilt removed, vs the noise) -> 4. tiles
# there (+ the insides of closed rings of tiles = features' flat tops); touching tiles = one
# REGION -> 5. one denser scan per region, its pattern from its shape -> 6. repeat at the
# new density inside the refined regions, until the finest spacing. Every scan and every
# move between scans starts and ends at rest (smooth motion, no jumps).
# who generates the scan paths:
cfg.grid_refine.scan_generator = "pyscanner_defined"   # "user_defined" = the package builds every waveform
                                               #   (constant tip speed, a level's tiles in one waveform);
                                               # "pyscanner_defined" = PyScanner's own scan types (raster 2,
                                               #   Lissajous 3, spiral 1) placed with its scan offsets, one
                                               #   do_scan per tile -- run the check in section 3b first
cfg.grid_refine.pyscanner_speed_basis = "peak"  # "pyscanner_defined": "peak" = never faster than tip_velocity_V_s,
                                               #   "mean" = tip_velocity_V_s on average (PyScanner's scans are sinusoidal)
cfg.grid_refine.pyscanner_raster_type = 2      # PyScanner raster_type (2 = sinusoidal)
# 1. sparse scan
cfg.grid_refine.sparse_pattern = "spiral"     # "raster", "lissajous" or "spiral"
cfg.grid_refine.sparse_crosshatch = False      # raster: second pass with lines the other way (a grid);
                                               #   False = ONE scan over the whole area
# 4. refinement
cfg.grid_refine.refine_pattern = "auto"       # "auto" = each region's pattern from its shape: round -> spiral
                                               #   (when shorter than a raster), rectangular -> auto_rect_pattern,
                                               #   irregular -> raster with every line clipped to the region;
                                               #   or force "raster", "lissajous" or "spiral" for every region
cfg.grid_refine.auto_rect_pattern = "raster"  # "auto": rectangular regions -- "raster" or "lissajous"
cfg.grid_refine.merge_tiles = True            # touching tiles (sides or corners) = one region, one scan
cfg.grid_refine.fill_interiors = True         # a closed ring of tiles (a feature's outline): scan its inside
                                               #   too -- measures the flat top, finds internal edges. Insides
                                               #   up to the largest feature wide (priors) -- bigger = substrate
cfg.grid_refine.fill_interior_levels = 1      # ... on the first N refinement levels only (later: edges only)
cfg.grid_refine.refine_factor = 3.0           # each level has lines this much closer
cfg.grid_refine.min_spacing_V = None          # final resolution = finest line spacing (None = width / 255,
                                               #   like a 256-line raster)
cfg.grid_refine.tile_size_lines = 2.0         # tile width, in line spacings of the level before (at most
                                               #   half the smallest feature -- priors)
cfg.grid_refine.raster_direction = "auto"     # "auto" (lines cross the edges), "x", "y" or "alternate"
cfg.grid_refine.batch_tiles = True            # "user_defined": all regions of a level in ONE waveform
                                               #   ("pyscanner_defined": one do_scan per region)
cfg.grid_refine.lissajous_phi_rad = None      # None = 0 (covers most evenly)
# 3. detection
cfg.grid_refine.detect_k = 5.0                # flag where local variation > k x noise -- higher = fewer tiles
cfg.grid_refine.detect_threshold = None       # or a fixed threshold (units of AI0) instead of the auto one
cfg.grid_refine.detect_on = "reconstruction"        # "measured" = decide from the measured pixels only (path independent
                                               #   of any reconstruction); "reconstruction" = inpaint first, decide
                                               #   on the inpainted image (finds structure predicted between lines)
cfg.grid_refine.detect_reconstruction = "linear"  # inpainting for "reconstruction": "linear", "fast",
                                               #   "compressed_sensing", "gp" or "live_view" (= cfg.live_view.reconstruction;
                                               #   CS/GP use the live-view parameters above and take longer per level)
# smooth motion + stopping
cfg.grid_refine.ease_time_s = 0.005           # speed-up / slow-down time at every scan start and end
cfg.grid_refine.max_levels = 2
cfg.grid_refine.max_run_time_s = None

# --- readback (real hardware only) -- Z/deflection input; there's no AO for Z, X/Y
# are the only outputs (AO_X_CONTROL/AO_Y_CONTROL in backends/pyscanner_backend.py) ---
cfg.readback.ai_channel = "AI0"  # CONFIRM ON HARDWARE which AI your deflection signal is wired to
cfg.readback.mode = "raw"  # "raw" (per-sample, CONFIRM ON HARDWARE) or "image" (binned)
# PyScanner fast rasters (grid_refine): the AI record lags/leads the path in time, so
# trace and retrace show every feature twice along the fast axis. "fit" = fit the
# shift on every fast raster (trace and retrace must agree) and re-place Z; "fixed" =
# always fast_raster_delay_s; "off" = none. fast_raster_delay_s is also the starting
# estimate for "fit" (+0.127 s measured on 2026-10-06)
cfg.readback.fast_raster_delay = "fit"
cfg.readback.fast_raster_delay_s = 0.127

# --- Asylum Research / Igor (section 3a and around every run cell) ---
cfg.asylum.controller = "cypher"     # X/Y loops read "Cypher.Input.A/B" ("arc": "Arc.Input.A/B")
cfg.asylum.crosspoint = {"InA": "BNCIn0", "InB": "BNCIn1"}   # FPGA AO0 / AO1 come in on BNCIn0 / BNCIn1
cfg.asylum.lock_crosspoint = True    # lock them, so a mode change doesn't re-route them
cfg.asylum.center_first = True       # setup: Go There (GoForce_1, spot 0) before handing X/Y to the FPGA
cfg.asylum.center_transit_s = 2.0     # time given to get there
cfg.asylum.engage_wait_s = 5.0       # time SimpleEngageMe gets to reach the surface
cfg.asylum.withdraw_wait_s = 2.0

# --- X/Y position sensors (optional): the real tip position, recorded with Z ---
# The controller's XSensor / YSensor are copied to Cypher BNCOut0 / BNCOut1 (a PIDS loop each
# -> a free DAC -> the BNC, locked), wired to PyScanner AI1 / AI2. The setup cell (3a) routes
# them and CALIBRATES them (tip withdrawn: a 3 x 3 grid of drive positions; AI V -> sensor V
# -> um with Igor's XLVDTSens / XLVDTOffset -> drive V); every run cell re-checks the routing
# (an Igor scan resets it). With a calibration, every image is built from the measured
# positions (piezo errors gone) and the axes are in um. Needs readback.mode == "raw".
cfg.position_sensors.enabled = True
cfg.position_sensors.x_ai_channel = "AI1"   # BNCOut0 (XSensor) wired here
cfg.position_sensors.y_ai_channel = "AI2"   # BNCOut1 (YSensor) wired here
cfg.position_sensors.x_loop = 3              # Cypher PIDS loops doing the copy (not 0/1: X/Y follow the FPGA)
cfg.position_sensors.y_loop = 4
cfg.position_sensors.x_bnc = "BNCOut0"
cfg.position_sensors.y_bnc = "BNCOut1"
cfg.position_sensors.calibration_span_V = 2.0    # calibration grid: +- this, in drive volts
cfg.position_sensors.calibration_settle_s = 0.3  # wait at each point before recording
cfg.position_sensors.x_m_per_V = None        # sensor V -> m; None = Igor's XLVDTSens (3.83 um/V on 2026-10-08)
cfg.position_sensors.y_m_per_V = None        #                      YLVDTSens (3.98 um/V)
cfg.position_sensors.smooth_samples = 0      # moving average over N samples (sensor noise); 0 = none
cfg.position_sensors.divide_output = True   # True: OutC/OutD = sensor / output_divider, so sensors past
cfg.position_sensors.output_divider = 2.0    #   +-10 V still fit the BNC / AI range (calibration absorbs it)
# with a good calibration (the setup cell prints "calibration NOT OK" otherwise -> all of these off):
cfg.position_sensors.plan_on_sensors = True   # every mode decides where to scan next from the MEASURED positions
cfg.position_sensors.correct_linear = True    # (a) waveforms sent through the inverse drive map (scale, cross-talk)
cfg.position_sensors.correct_offset = True    # (b) each waveform's measured end error shifts the next ones
cfg.position_sensors.correction_gain = 0.5    #     (b) fraction of a new error taken each time (1 = all of it)
cfg.position_sensors.correction_max_V = 1.0   #     (b) never more than this (drive volts)

# --- Z signal on a Cypher BNC (cfg.z_signal; asylum.AsylumControl.route_z, setup step 3d) ---
# PyScanner's AI0 (cfg.readback.ai_channel) stays cabled to z_signal.bnc; the setup copies the
# chosen Z signal there through a Cypher loop and checks (tip withdrawn) that AI0 reads it
cfg.z_signal.route = False           # True: AI0's cable on BNCOut2, signal chosen below
cfg.z_signal.source = "ZSensor"        # "ZSensor" (Z LVDT) or the Z piezo's Cypher signal name (CONFIRM)
cfg.z_signal.loop = 5                  # Cypher PIDS loop doing the copy (3/4: X/Y sensors)
cfg.z_signal.dac = "OutFastA"          # preferred DAC (another free one if taken; OutA/B = Igor's loops 0/1)
cfg.z_signal.bnc = "BNCOut2"
cfg.z_signal.output_divider = 1.0      # BNC = signal / this (BNC / AI stop at +-10 V)
cfg.z_signal.invert = True             # flip the sign in software: ZSensor reads opposite to the height
cfg.z_signal.units = "nm"              # "nm": Z converted to nm as it comes in (needs route + ZSensor's
                                       #   ZLVDTSens): plots, saved z AND thresholds in nm, e.g.
                                       #   cfg.edge_explorer.edge_detect.manual_threshold = 100.0  # nm

# --- safety ---
cfg.safety.max_amplitude_V = 10.0   # spirals are moved/shrunk to stay inside this; circles stop at it
cfg.safety.max_samples_per_waveform = 4_000_000  # longer waveforms go to PyScanner in pieces of this size
                                                   # (3.8 M worked on hardware, 8.5 M did not)
cfg.safety.clamp_mode = "raise"  # keep "raise" for real runs -- see README

# --- live view (wall-clock throttled -- see LiveViewConfig) ---
cfg.live_view.min_redraw_interval_s = 0.25
cfg.live_view.min_reconstruction_interval_s = 1.0
cfg.live_view.figsize = (13.5, 4.0)  # inches, 3 columns -- shrink this (e.g. (10, 3)) if it's
                                       # still wider than your notebook window
# --- reconstruction (edge-follow live view: col 1 = scan path, col 2 = binned AI0
# exactly like PyScanner's image_AI0, col 3 = col 2's empty pixels filled in) ---
cfg.live_view.reconstruction = "compressed_sensing"  # "compressed_sensing", "gp", "linear", "fast" or "interpolated"
                                          #   "linear" = interpolation between ALL measured pixels (the old
                                          #   "interpolated" only used 4000 subsampled points) -- best on steps
                                                     #   ("fast": local average -- no CS ghosts on sparse grid_refine data)
cfg.live_view.n_image_pixels = 200     # binned/reconstructed image size -- match PyScanner's N_image_pixels
# compressed sensing (image assumed sparse in the 2D DCT; ~0.2 s per update)
cfg.live_view.cs_lambda = 0.005       # higher = smoother, lower = sticks closer to the measured pixels (was 0.5)
cfg.live_view.cs_iterations = 500      # more = more converged, slower (~0.2 s per update at 500)
cfg.live_view.cs_detrend = True        # remove the sample tilt (best-fit plane) before CS -- sharper features
# GP (RBF + noise kernel; ~1 s per update with gp_optimize=False)
cfg.live_view.gp_kernel = "matern52"   # "matern52" (as in the SS-KPFM paper, no ringing at edges) or "rbf"
cfg.live_view.gp_max_points = 1500     # measured pixels the GP trains on -- the speed knob (cost ~ n^3)
cfg.live_view.gp_length_scale_frac = 0.05  # feature size, as a fraction of the image width
cfg.live_view.gp_optimize = False      # True = fit the kernel each update: better, but seconds-to-tens-of-
                                       # seconds per update -- raise min_reconstruction_interval_s if you do
LIVE_VIEW_EXTENT_V = 0.45  # starting/fallback view size ONLY, for before any data exists
                            # (and the seed for cell 5's saved reconstruction if the run
                            # somehow has zero points) -- once the scan has points, every
                            # panel's axes auto-fit to wherever the tip has actually been
                            # instead (see viz.bounds_from_history), so this doesn't need
                            # to match your actual search/path radii

# =====================================================================================
# PFM (cfg.pfm; selfdriving_afm/pfm.py) -- Asylum in PFM mode, the drive on while engaged
# =====================================================================================
cfg.pfm.enabled = True
cfg.z_signal.route = False               # no Z copy here: Cypher loop 5 carries the PFM amplitude
# --- PFM amplitude -> AI0 ---
cfg.pfm.amplitude.enabled = True         # False: phase only
cfg.pfm.amplitude.source = "Amplitude"   # Igor's signal (ir_ReadValue("Amplitude") must give a number)
cfg.pfm.amplitude.ai_channel = "AI0"
cfg.pfm.amplitude.side = "Cypher"        # whose PIDS loop / DAC / BNC: "Cypher" or "ARC"
cfg.pfm.amplitude.loop = 5
cfg.pfm.amplitude.dac = "OutFastA"
cfg.pfm.amplitude.bnc = "BNCOut2"
cfg.pfm.amplitude.gain = 5000           # BNC volts per volt of amplitude (keep gain x amplitude < 10 V)
# --- PFM phase -> AI3 ---
cfg.pfm.phase.enabled = True            # False: amplitude only
cfg.pfm.phase.source = "Phase"
cfg.pfm.phase.ai_channel = "AI3"
cfg.pfm.phase.side = "ARC"               # the Cypher's loops are all taken -- the ARC's 2..5 are free
cfg.pfm.phase.loop = 3
cfg.pfm.phase.dac = "OutA"               # CONFIRM ON HARDWARE (the ARC's DAC names)
cfg.pfm.phase.bnc = "BNCOut0"            # the ARC's BNC output cabled to AI3 -- CONFIRM ON HARDWARE
cfg.pfm.phase.gain = 0.05                # BNC volts per degree: +-180 deg -> +-9 V
# --- what a domain wall is (pfm.DomainModel) ---
cfg.pfm.walls.min_phase_separation_deg = 60.0   # two domains = phases at least this far apart
cfg.pfm.walls.amp_dip_fraction = 0.6     # a wall: amplitude below this x the domains'
cfg.pfm.walls.confirm_with_amplitude = False    # (both signals) a phase flip counts only with an amplitude dip
cfg.pfm.walls.confirm_window_V = 0.05    # ...within this much path of it
cfg.pfm.walls.phase_memory = 0.8         # the domains' phases follow drift (0 = only the last waveform)
cfg.edge_explorer.edge_detect.hysteresis_fraction = 0.4   # against flicker at a wall
cfg.edge_explorer.edge_detect.min_run_V = 0.004            # shorter flips = noise
# colour maps: the live view shows the phase if it's on (else the amplitude)
PFM_CMAP = {"amplitude": "viridis", "phase": "inferno"}
cfg.live_view.cmap = PFM_CMAP["phase"] if cfg.pfm.phase.enabled else PFM_CMAP["amplitude"]

## 3. Backend: PyScanner (or the simulation)

Created here, before the setup, so the setup (3a) can calibrate the position sensors
**before** the priors are asked (3b) -- with the sensors on, the priors are in nm and need
that calibration.

In [ ]:
if USE_HARDWARE:
    backend = PyScannerBackend(vi, cfg.safety, cfg.readback, position_sensors=cfg.position_sensors)
    ground_truth_fn = None
else:
    from selfdriving_afm import pfm as _pfm
    # simulated PFM: "bubbles", "stripes" or "labyrinth" domains, walls ~0.03 V wide
    backend = _pfm.PFMSampleBackend(cfg.pfm, _pfm.domain_field("bubbles", seed=3), seed=1)
    ground_truth_fn = None


## 3a. Microscope setup: centre, AO0/AO1 at 0, crosspoint, engage

Run **once, before any scan** (`selfdriving_afm/asylum.py`). Prints every step and every
Igor command it sends:

0. withdraw (nothing moves laterally while engaged)
1. tip to the **centre of the image** -- Igor's Go There to spot 0, `DoForceFunc("GoForce_1")` (skipped
   if the X/Y loops already follow the FPGA in this session)
2. checks the FPGA's **AO0/AO1 are at 0** -- if not, moves them there smoothly (PyScanner's
   line-scan move, at the scan tip speed)
3. **crosspoint**: InA <- BNCIn0, InB <- BNCIn1, locked and written; then the X/Y piezo
   loops follow inputs A/B (`ir_SetPISLoop`, the Cypher lines by default -- `cfg.asylum`)
   - with `cfg.position_sensors.enabled`: **XSensor/YSensor -> BNCOut0/1** (-> PyScanner AI1/AI2),
     then the **sensor calibration** (tip withdrawn: a 3 x 3 grid of drive positions, printed --
     check the AI gain is ~1 and the um per drive volt)
4. **engage** (`SimpleEngageMe`)

Every run cell below (3d, 4, 6) first **checks the routing** (X/Y -> FPGA, and the sensors)
and redoes it if Igor reset it (an Igor scan does), runs **engaged** (it engages again if needed), and
**withdraws and goes back to (0, 0) smoothly when it ends** -- also if it fails or you
interrupt it. If a cell is killed so hard that this didn't happen, run **4b**.
Without hardware (`USE_HARDWARE = False`) the Igor commands are printed, not sent.

It runs **before the priors** (3b): with `cfg.position_sensors.enabled` the priors are asked
in nm, converted to drive volts with the calibration measured here.

In [ ]:
RUN_SETUP = True   # <-- False to skip (e.g. already set up and engaged in this session)

if RUN_SETUP or "afm" not in globals() or afm.backend is not backend:
    afm = asylum.AsylumControl(igor, backend, cfg)   # the executor joins in 3c (afm.run passes it on)
if RUN_SETUP:
    afm.setup()

In [ ]:
for k in range(8):
    try:
        g = afm.read_loop(k, side="Cypher")
        print(k, {key: g.get(key) for key in ("InputChannel", "OutputChannel", "StartEvent")})
    except Exception as e:
        print(k, "->", e)

for k in range(8):
    try:
        g = afm.read_loop(k, side="ARC")
        print(k, {key: g.get(key) for key in ("InputChannel", "OutputChannel", "StartEvent")})
    except Exception as e:
        print(k, "->", e)

## 3b. Priors -- what you expect on this sample

Asked **every run, before the scan starts** (press Enter to keep the value in brackets):
- the **smallest and largest expected domain** -- its narrowest width (a bubble's diameter,
  a stripe's width): the explorer sweeps ~the smallest apart, so none is missed;
- the **most the scan may explore**: a square (half-width and center).

**Units:** with the position sensors on (`cfg.position_sensors.enabled`, calibrated in 3a) in
**nm** along the sensors' axes, the centre in the sensors' coordinates (the "X/Y (um, sensor)"
axes of the plots) -- converted to drive volts with the setup's calibration (printed, with
the nm per drive volt); everything after that is in volts as before. Sensors off: in **V**.

They set every mode's area and size parameters (`selfdriving_afm/priors.py`) -- these
are no longer set in the configuration cell:
- **edge_follow**: the first search spiral covers the whole area, its turns half the
  smallest feature apart (no feature fits between two turns); later spirals fill what's
  left, inside the area. Tracing circles are sized from the curvature each circle
  measures (the angle its on-feature arc spans), between 0.25 x the smallest and
  0.75 x the largest feature -- small round features get small circles, long straight
  edges big ones (each circle costs ~3 s of do_scan overhead).
- **grid_refine**: the sparse scan covers the area, lines 0.7 x the smallest feature apart.
- **edge_explorer**: explores the area from its center, sweeps ~the smallest feature apart;
  follows each edge it crosses until it closes, then scans the inside with lines 0.7 x the
  smallest feature apart (holes found there are followed too).
- **live view**: always shows the whole area to explore, in every frame.
- **gp_path**: stays inside the area; GP length scale from the feature sizes.

The printout says what was set, with the first spiral's size and time.


In [ ]:
from selfdriving_afm import priors

# --- the defaults offered when asked (Enter keeps them) ---
PRIOR_DEFAULTS = priors.ScanPriors(
    feature_min_V=0.5,          # smallest expected DOMAIN: its narrowest width (V)
    feature_max_V=2.0,          # largest expected domain (V)
    area_half_width_V=4.0,      # max area to explore / discover: a square of this half-width (V)...
    area_center_V=(0.0, 0.0),   # ...centered here (V)
)
# with the position sensors on: asked in nm (sensor frame). None = PRIOR_DEFAULTS converted with the
# setup's calibration; or e.g. dict(feature_min_nm=2000, feature_max_nm=6000, area_half_width_nm=15000,
# area_center_nm=(0, 0))  -- the centre in the sensors' coordinates (the plots' um axes x 1000)
PRIOR_DEFAULTS_NM = None
ASK_PRIORS = True               # False = use the defaults without asking

PRIORS = priors.ask_for_run(cfg, PRIOR_DEFAULTS, PRIOR_DEFAULTS_NM, sensor_cal=getattr(afm, "sensor_cal", None),
                            interactive=ASK_PRIORS, what="domain")
print("Priors:", PRIORS)
for note in priors.apply(cfg, PRIORS):   # sets every mode's area + size parameters in cfg
    print(" ", note)
LIVE_VIEW_EXTENT_V = PRIORS.area_half_width_V
# the live view always shows this whole area (cfg.live_view.fixed_bounds_V, set by the priors);
# cfg.live_view.fixed_bounds_V = None here to let it grow with the scanned area instead


## 3c. Build executor + planner + live view

In [ ]:
# executor + planner + live view for MODALITY. This is the SAME code Simulation.ipynb
# runs (selfdriving_afm/modes.py) -- only the backend differs -- so a mode behaves here
# exactly as it did in simulation. New modes are added there once, for both notebooks.
session = modes.build(MODALITY, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V,
                      ground_truth_fn=ground_truth_fn, priors=PRIORS)
executor, planner, live = session.executor, session.planner, session.live

# Building the live view above pushed an initial (empty) frame into THIS cell's own
# output, just to confirm the view was built correctly. The actual live view while
# scanning shows up in cell 4's OWN output instead (it gets replaced in place there,
# once per redraw) -- not here -- since each cell has its own separate output area.
# See viz._LiveViewBase._redraw's docstring for why it's done this way rather than
# via the interactive ipympl canvas's own redraw.
afm.executor = executor   # the setup's calibration and Z routing reach it through the backend / afm.run


## 3d. Check PyScanner's own scan types (once, on the instrument)

Only needed for `cfg.grid_refine.scan_generator = "pyscanner_defined"`. Set `RUN_PROBE = True`
and run this cell: it runs a few **small** PyScanner scans (a raster, a rotated raster, a
Lissajous, a spiral -- 0.2 V squares around (0.1, 0.05) V, at the tip speed set above) and prints, for
each, the sample count vs duration, where it actually started vs where the package moved
the tip, whether `AO_vs_time` includes the scan offset, and the wall time per scan. It also
prints the spiral/raster/Lissajous parameter clusters. It saves the printout and every scan's path (spiral included) to
`scan_data/pyscanner_probe_<time>.npz` and `.txt`.

In [ ]:
RUN_PROBE = True  # <-- True to run the check (moves the tip: small scans around (0.1, 0.05) V)

if RUN_PROBE:
    from selfdriving_afm.pyscanner_scans import probe_pyscanner_scans
    with afm.run("PyScanner scan-type check"):   # withdraws + back to (0, 0) afterwards
        probe_reports = probe_pyscanner_scans(executor, center_xy=(0.1, 0.05), size_V=0.2, lines=10)
    executor.reset(cfg.execution.start_xy_V)  # the probe isn't part of the run


## 4. Run

This cell blocks until the scan finishes (or you interrupt the kernel). The live
view appears **in this cell's own output** (below), replaced in place each time a
segment comes back -- not in cell 3 above, which only shows the initial empty frame.

Suggested first real-hardware test: small `max_search_radius_V` well inside a known
safe area, `max_features=1`, on a region you already know the topography of, so you
can sanity-check the traced boundary before trusting the planner on an unknown region.


In [ ]:
# runs the planner from cfg.execution.start_xy_V, finalizes the live view and prints
# the mode's summary (modes.MODES[MODALITY].summary) -- same as in Simulation.ipynb.
# Engaged for the run; withdraws and goes back to (0, 0) smoothly when it ends (also
# if it fails or is interrupted) -- see section 3a
with afm.run(MODALITY, executor):
    result = modes.run(session)
# full scan-by-scan log (spirals, circles, moves): print("\n".join(live.log))


## 4b. Withdraw and return the tip to (0, 0)

**Withdraws**, then moves the FPGA's AO0/AO1 smoothly back to **(0, 0)** at the scan tip
speed (`cfg.velocity.tip_velocity_V_s`), via PyScanner's line-scan move
(`line_scan_control_cluster` / `do_probe_move`). The run cells already do this when they
end; run this one if you killed a run cell in the middle (e.g. to debug) and it didn't.
It doesn't touch `executor.history`, so the save cell (5) still saves the run.


In [ ]:
afm.finish()   # withdraw, then AO0/AO1 smoothly to (0, 0)


## 5. Save results

Saved into `OUTPUT_DIR` (cell 2), as one `.npz` + one `.png` per run:
- **The trajectory scanned** -- `x_cmd`/`y_cmd`/`z` (commanded X/Y, measured Z, in
  acquisition order), `kinds` (which primitive each stretch came from), and
  `x_sensor`/`y_sensor` too if `cfg.position_sensors.enabled`.
- **The reconstructed data itself**, not just a picture of it -- `recon_X`/`recon_Y`
  plus `recon_Z` (edge-follow) or `recon_mean`/`recon_sigma` (GP path), the actual
  arrays the live view was plotting. Reload these later (or recompute them with
  different parameters) in `Analyze_SelfDriving_Scan.ipynb`.
- `points` -- the same x/y/z used for the reconstruction above (sensor-corrected if
  position sensors were on, otherwise identical to `x_cmd`/`y_cmd`/`z`).
- The final live-view figure, as a `.png`.


In [ ]:
# trajectory + reconstruction (+ grid_refine maps and tiles) -> OUTPUT_DIR, in the format
# Analyze_SelfDriving_Scan.ipynb reads -- see modes.save
out_prefix = modes.save(session, OUTPUT_DIR, prefix="scan", fallback_extent=LIVE_VIEW_EXTENT_V)


## 6. Reference raster with the same number of pixels (optional)

Run **after** the save cell (5): acquires a raster with **PyScanner's own fast raster**
(scan_type 2, the "fast raster scan" tab) over the **same area and pixel grid** as the run
just saved, with as many lines as it takes to measure **about as many pixels as the run did**
(one line = one pixel row; the lines spread evenly over the area, first and last on its
edges) -- the classic scan you could have done with the same budget. It's added to the
**same .npz** (`raster_*` arrays), so `Analyze_SelfDriving_Scan.ipynb` can compare the two.

- Set `RUN_REFERENCE_RASTER = True` to acquire it -- with `False` the cell only prints the
  plan (lines, spacing, speed, number of PyScanner scans, time) so you can decide first.
- It's **one** fast raster (one do_scan), exactly like setting the "fast raster scan" tab
  by hand: `raster_scan_size_x/y_V` = the run's area, `raster_N_scan_lines` = the lines
  needed to match the run's measured pixels (on its 200 x 200 grid), `raster_scan_duration_s`
  = `RASTER_DURATION_S` (`None` = 1 s per line, e.g. 20 lines in 20 s), `raster_type = 2`
  (sinusoidal), `scan_x/y_offset_V` = the area's centre, and for lines along y
  `scan_rotation_deg = 90`. Offsets, rotation and `scan_type` go back to 0 afterwards.
- The tip moves to the start smoothly and goes back to (0, 0) afterwards; the run's own
  data is untouched. The .npz is rewritten with the raster added (via a temporary file);
  the samples are kept thinned (a quarter pixel apart), the binned image and its
  reconstruction (`raster_columns`) in full.
- With the **position sensors** on, the raster also records them (the same AI record as Z) and
  is saved a second time **at the measured positions** (`raster_*_sensor`, `raster_xs/ys/zs`):
  along each line the sensors' position, each line at its measured place. The plot and the
  Analyze notebook use that one; the commanded one (time-shift fitted) is kept too.

In [ ]:
RUN_REFERENCE_RASTER = True   # <-- True to acquire it (moves the tip over the whole area)
RASTER_FAST_AXIS = "x"         # lines along "x" or "y" (y = PyScanner's raster rotated 90 deg)
RASTER_N_LINES = None          # None = as many lines as it takes to match the run's measured pixels
RASTER_DURATION_S = None       # raster_scan_duration_s of the fast raster; None = 1 s per line

from selfdriving_afm import reference_raster
cfg.velocity.tip_velocity_V_s = 100

RUN_NPZ = f"{out_prefix}.npz"   # the run saved by the cell above
raster_plan = reference_raster.plan(RUN_NPZ, cfg, fast_axis=RASTER_FAST_AXIS, n_lines=RASTER_N_LINES,
                                    duration_s=RASTER_DURATION_S)
print(reference_raster.describe(raster_plan))
print("   ", raster_plan["scans"][0].describe())
if RUN_REFERENCE_RASTER:
    # with cfg.pfm on, the raster also records every PFM signal that's on (amplitude / phase) and
    # saves it into the same .npz (raster_ch_<signal>_*), commanded + measured positions
    with afm.run("reference raster"):   # engaged; withdraws + back to (0, 0) afterwards
        raster_ref = reference_raster.acquire(raster_plan, backend, cfg, tip_xy=executor.last_pos,
                                              return_to_origin=False, sensor_cal=getattr(afm, "sensor_cal", None))
    reference_raster.save_into(RUN_NPZ, raster_ref)
    reference_raster.plot(RUN_NPZ, cmap=cfg.live_view.cmap)    # z, then every PFM signal
else:
    print("RUN_REFERENCE_RASTER is False -- nothing acquired")